In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
base_dir = Path("D:/Mitosis WSI CCMCT")
clean_paths = defaultdict(list)

clean_paths = defaultdict(list)

with open(base_dir / "clean_paths.txt", "r") as f:
    for line in f:
        label, filename = line.strip().split(",")
        clean_paths[label].append(base_dir / ("mitotic" if label == "2" else "non_mitotic") / filename)

for label in clean_paths:
    print(f"Class {label}: {len(clean_paths[label])}")

Class 1: 13611
Class 2: 11954
Class 3: 21995
Class 4: 18127
Class 7: 1640


In [4]:
mitotic = [(p, 1) for p in clean_paths["2"]]
non_mitotic = [(p, 0) for p in clean_paths["1"] + clean_paths["3"] + clean_paths["7"]]

print(f"Mitotic: {len(mitotic)}")
print(f"Non-mitotic: {len(non_mitotic)}")
print(f"Total: {len(mitotic) + len(non_mitotic)}")
print(f"Imbalance: {len(non_mitotic) / len(mitotic):.1f}:1")

Mitotic: 11954
Non-mitotic: 37246
Total: 49200
Imbalance: 3.1:1


In [5]:
class MitosisDataset(Dataset):

    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        return img, label

In [6]:
# transform = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], 
#                          std=[0.229, 0.224, 0.225])
# ])


train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [7]:
df_annots = pd.read_csv(base_dir / "meta_data" / "Annotations.csv")
uid_to_slide = dict(zip(df_annots["uid"], df_annots["slide"]))

all_data = mitotic + non_mitotic

slide_groups = defaultdict(list)

for path, label in all_data:
    uid = int(path.stem.split("_")[0])
    slide = uid_to_slide.get(uid)
    if slide is not None:
        slide_groups[slide].append((path, label))

print(f"Slides found: {len(slide_groups)}")
for slide in sorted(slide_groups.keys()):
    mit = sum(1 for _, l in slide_groups[slide] if l == 1)
    non = sum(1 for _, l in slide_groups[slide] if l == 0)
    print(f"  Slide {slide:2d}: {mit:5d} mitotic, {non:5d} non-mitotic, {mit+non:5d} total")

Slides found: 21
  Slide  4:  2875 mitotic,  6470 non-mitotic,  9345 total
  Slide  7:   948 mitotic,  3485 non-mitotic,  4433 total
  Slide  8:    54 mitotic,   237 non-mitotic,   291 total
  Slide 12:   847 mitotic,  3351 non-mitotic,  4198 total
  Slide 13:   685 mitotic,  2306 non-mitotic,  2991 total
  Slide 14:  1183 mitotic,  1708 non-mitotic,  2891 total
  Slide 15:   188 mitotic,   885 non-mitotic,  1073 total
  Slide 17:   371 mitotic,  1774 non-mitotic,  2145 total
  Slide 19:  2184 mitotic,  2402 non-mitotic,  4586 total
  Slide 21:  2547 mitotic,  3672 non-mitotic,  6219 total
  Slide 22:    26 mitotic,   644 non-mitotic,   670 total
  Slide 23:     0 mitotic,  1328 non-mitotic,  1328 total
  Slide 24:     2 mitotic,   748 non-mitotic,   750 total
  Slide 25:     7 mitotic,  1077 non-mitotic,  1084 total
  Slide 26:     1 mitotic,   323 non-mitotic,   324 total
  Slide 28:     0 mitotic,   606 non-mitotic,   606 total
  Slide 29:     7 mitotic,  1940 non-mitotic,  1947 tot

In [8]:
train_slides = [4, 7, 12, 13, 17, 15, 23, 25, 28, 29, 32, 34, 35, 36]
val_slides =   [14, 8, 22, 24]
test_slides =  [19, 21, 26]

train_data = []
val_data = []
test_data = []

for slide in train_slides:
    train_data.extend(slide_groups[slide])
for slide in val_slides:
    val_data.extend(slide_groups[slide])
for slide in test_slides:
    test_data.extend(slide_groups[slide])

for name, data in [("Train", train_data), ("Val", val_data), ("Test", test_data)]:
    mit = sum(1 for _, l in data if l == 1)
    non = sum(1 for _, l in data if l == 0)
    print(f"{name:5s} | Mitotic: {mit:5d} | Non-mitotic: {non:5d} | Total: {mit+non:5d} | Ratio: {non/max(mit,1):.1f}:1")

Train | Mitotic:  5957 | Non-mitotic: 27512 | Total: 33469 | Ratio: 4.6:1
Val   | Mitotic:  1265 | Non-mitotic:  3337 | Total:  4602 | Ratio: 2.6:1
Test  | Mitotic:  4732 | Non-mitotic:  6397 | Total: 11129 | Ratio: 1.4:1


In [9]:
import random

random.shuffle(train_data)

train_dataset = MitosisDataset(train_data, transform=train_transform)
val_dataset = MitosisDataset(val_data, transform=val_transform)
test_dataset = MitosisDataset(test_data, transform=val_transform)

train_labels = [label for _, label in train_data]
class_counts = [train_labels.count(0), train_labels.count(1)]
weights = [1.0 / class_counts[label] for label in train_labels]

sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights))

train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 523
Val batches: 72
Test batches: 174


In [10]:
model = models.resnet18(pretrained=True)

# for param in model.parameters():
#     param.requires_grad = False

# for param in model.layer4.parameters():
#     param.requires_grad = True

model.fc = nn.Linear(512, 1)

model = model.to(device)
print(model.fc)

C:\Users\user\.conda\envs\gpu_env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\user\.conda\envs\gpu_env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Linear(in_features=512, out_features=1, bias=True)


In [11]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
# optimizer = optim.Adam([
#     {"params": model.layer4.parameters(), "lr": 1e-4},
#     {"params": model.fc.parameters(), "lr": 1e-3}
# ])

In [12]:
num_epochs = 15

for epoch in range(num_epochs):
    
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        
        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()                                    
        optimizer.step()                                  
        running_loss += loss.item()  
        preds = (torch.sigmoid(outputs) > 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).long()
            val_correct += (preds == labels.long()).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}")

Epoch 1/15 | Train Loss: 0.6621 Acc: 0.604 | Val Loss: 0.7059 Acc: 0.486
Epoch 2/15 | Train Loss: 0.6419 Acc: 0.627 | Val Loss: 0.7023 Acc: 0.521
Epoch 3/15 | Train Loss: 0.6320 Acc: 0.638 | Val Loss: 0.6601 Acc: 0.611
Epoch 4/15 | Train Loss: 0.6259 Acc: 0.635 | Val Loss: 0.6670 Acc: 0.593
Epoch 5/15 | Train Loss: 0.6196 Acc: 0.646 | Val Loss: 0.6696 Acc: 0.594
Epoch 6/15 | Train Loss: 0.6165 Acc: 0.646 | Val Loss: 0.7285 Acc: 0.517
Epoch 7/15 | Train Loss: 0.6117 Acc: 0.656 | Val Loss: 0.6866 Acc: 0.576
Epoch 8/15 | Train Loss: 0.6065 Acc: 0.662 | Val Loss: 0.6956 Acc: 0.617
Epoch 9/15 | Train Loss: 0.5974 Acc: 0.666 | Val Loss: 0.7360 Acc: 0.537
Epoch 10/15 | Train Loss: 0.5934 Acc: 0.670 | Val Loss: 0.6929 Acc: 0.606
Epoch 11/15 | Train Loss: 0.5819 Acc: 0.683 | Val Loss: 0.7004 Acc: 0.617
Epoch 12/15 | Train Loss: 0.5778 Acc: 0.685 | Val Loss: 0.7168 Acc: 0.568
Epoch 13/15 | Train Loss: 0.5712 Acc: 0.691 | Val Loss: 0.6854 Acc: 0.618
Epoch 14/15 | Train Loss: 0.5625 Acc: 0.702 | V

In [13]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze()
        preds = (torch.sigmoid(outputs) > 0.5).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=["Non-mitotic", "Mitotic"]))
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

 Non-mitotic       0.59      0.47      0.52      6397
     Mitotic       0.44      0.56      0.49      4732

    accuracy                           0.51     11129
   macro avg       0.51      0.51      0.51     11129
weighted avg       0.52      0.51      0.51     11129

Confusion Matrix:
[[2981 3416]
 [2078 2654]]
